## 1. 必要なライブラリのインポート

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# vieworcaのインポート
try:
    from vieworca import vieworca

    print("vieworcaをインポートしました")
except ImportError:
    print("vieworcaをインストール中...")
    import subprocess

    subprocess.check_call(["pip", "install", "vieworca"])
    from vieworca import vieworca

    print("vieworcaをインストールしてインポートしました")

vieworcaをインストール中...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 6.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [vieworca]1/2 [vieworca]


ModuleNotFoundError: No module named 'vieworca'

## 2. ORCAログファイルの解析

In [2]:
# IRCログファイルのパス
log_file = "irc.log"

# vieworcaでログファイルを解析
try:
    orca = vieworca(log_file)
    print(f"✓ IRCログファイルを解析しました: {log_file}")
    print(f"\n計算の概要:")
    print(f"  ジョブタイプ: {orca.jobtype}")
    print(f"  収束状況: {orca.scf_converged}")
except Exception as e:
    print(f"エラー: {e}")
    orca = None

エラー: name 'vieworca' is not defined


## 3. IRC最終構造の情報取得

In [ ]:
if orca:
    # エネルギー情報
    try:
        final_energy = orca.scf_energy
        print(f"最終エネルギー: {final_energy:.6f} Eh")
    except:
        print("エネルギー情報を取得できませんでした")

    # 原子情報
    try:
        atoms = orca.atoms
        coords = orca.coords
        print(f"\n原子数: {len(atoms)}")
        print(f"\n原子リスト (最初の10個):")
        for i, atom in enumerate(atoms[:10]):
            print(f"  {i+1}: {atom}")
    except:
        print("原子情報を取得できませんでした")

    # 振動数情報
    try:
        if hasattr(orca, "vibfreqs"):
            print(f"\n振動数数: {len(orca.vibfreqs)}")
            print(
                f"最低振動数 (虚数モード): {orca.vibfreqs[0]:.2f} cm⁻¹"
                if orca.vibfreqs[0] < 0
                else f"最低振動数: {orca.vibfreqs[0]:.2f} cm⁻¹"
            )
    except:
        print("振動数情報は利用できません")
else:
    print("ログファイルが解析できなかったため、情報取得をスキップします")

ログファイルが解析できなかったため、情報取得をスキップします


## 4. IRCトラジェクトリの詳細分析

In [ ]:
# ORCAのproperty.txtファイルを解析
property_file = "irc.property.txt"

if Path(property_file).exists():
    print(f"Property情報の読み込み: {property_file}")
    with open(property_file, "r") as f:
        content = f.read()
        # 最初の50行を表示
        lines = content.split("\n")
        print(f"\nプロパティファイル内容 (最初の30行):")
        for line in lines[:30]:
            if line.strip():
                print(line)
else:
    print(f"Property情報ファイルが見つかりません")

Property情報の読み込み: irc.property.txt

プロパティファイル内容 (最初の30行):
*************************************************
******************* ORCA 6.1.1 ******************
*************************************************
$Calculation_Status
   &GeometryIndex 1
   &version [&Type "String"] "6.1.1"
   &progName [&Type "String"] "LeanSCF"
   &Status [&Type "String"] "NORMAL TERMINATION"
$End
$Geometry
   &GeometryIndex 1
   &NAtoms [&Type "Integer"] 14
   &NCorelessECP [&Type "Integer"] 0
   &NGhostAtoms [&Type "Integer"] 0
   &CartesianCoordinates [&Type "Coordinates", &Dim(14,4), &Units "Bohr"] 
              O      3.014522157836   -0.442882396387   -1.322972480613
              N     -3.854990777766   -0.239546220588   -0.194893282183
              C     -1.453105614903   -0.284347854551   -1.545888020231
              C      0.809782396742   -0.424269680249    0.206366767521
              H     -1.333387218397    1.413636306238   -2.700058418900
              H     -1.456869254822   -1.90579648474

## 5. IRC前向き・後向きパスの分析

In [ ]:
# 前向きパス (IRC_F) の情報
print("=== 前向きパス (IRC_F) の分析 ===")

for irc_file in ["irc_IRC_F.log", "irc_IRC_F.xyz"]:
    if Path(irc_file).exists():
        print(f"\n✓ {irc_file} が見つかりました")
    else:
        print(f"✗ {irc_file} は見つかりません")

print("\n=== 後向きパス (IRC_B) の分析 ===")

for irc_file in ["irc_IRC_B.log", "irc_IRC_B.xyz"]:
    if Path(irc_file).exists():
        print(f"✓ {irc_file} が見つかりました")
    else:
        print(f"✗ {irc_file} は見つかりません")

## 6. 構造変化の解析

In [ ]:
# トラジェクトリファイルから構造情報を抽出
def parse_xyz_file(filepath):
    """XYZファイルから構造情報を抽出"""
    structures = []
    energies = []

    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if lines[i].strip().isdigit():
            natoms = int(lines[i].strip())

            # ヘッダーからエネルギーを抽出
            if i + 1 < len(lines):
                header = lines[i + 1]
                match = re.search(r"E\s+([-\d.]+)", header)
                if match:
                    energies.append(float(match.group(1)))

                # 原子座標を抽出
                atoms = []
                for j in range(natoms):
                    if i + 2 + j < len(lines):
                        parts = lines[i + 2 + j].split()
                        if len(parts) >= 4:
                            atoms.append(parts[0])  # 元素記号
                structures.append(atoms)

            i += natoms + 2
        else:
            i += 1

    return structures, np.array(energies)


# 全体トラジェクトリの分析
traj_file = "irc_IRC_Full_trj.xyz"
if Path(traj_file).exists():
    structures, energies = parse_xyz_file(traj_file)
    print(f"トラジェクトリファイル分析: {traj_file}")
    print(f"  総ステップ数: {len(structures)}")
    print(f"  各ステップの原子数: {len(structures[0]) if structures else 0}")
    print(f"\nエネルギー統計:")
    print(f"  最小エネルギー: {energies.min():.6f} Eh")
    print(f"  最大エネルギー: {energies.max():.6f} Eh")
    print(f"  エネルギー差: {(energies.max() - energies.min()) * 2625.5:.2f} kJ/mol")
    print(f"  平均エネルギー: {energies.mean():.6f} Eh")
else:
    print(f"トラジェクトリファイル {traj_file} が見つかりません")

## 7. IRC パス分析サマリー

In [ ]:
print("\n" + "=" * 60)
print("IRC計算 結果サマリー")
print("=" * 60)

if Path(traj_file).exists():
    # 反応物側（最初）と生成物側（最後）の情報
    print(f"\n反応物側 (ステップ 1):")
    print(f"  エネルギー: {energies[0]:.6f} Eh")
    print(f"  相対エネルギー: 0.00 kJ/mol")

    print(f"\n遷移状態 (最低エネルギー):")
    ts_idx = np.argmin(energies)
    print(f"  ステップ: {ts_idx + 1}")
    print(f"  エネルギー: {energies[ts_idx]:.6f} Eh")
    print(
        f"  相対エネルギー: {(energies[ts_idx] - energies.min()) * 2625.5:.2f} kJ/mol"
    )

    print(f"\n生成物側 (最後のステップ):")
    print(f"  エネルギー: {energies[-1]:.6f} Eh")
    print(f"  相対エネルギー: {(energies[-1] - energies.min()) * 2625.5:.2f} kJ/mol")

    print(f"\nIRC計算の統計:")
    print(f"  総ステップ数: {len(energies)}")
    print(
        f"  反応物 → 生成物のエネルギー変化: {(energies[-1] - energies[0]) * 2625.5:.2f} kJ/mol"
    )
    print(f"  最小エネルギーへの収束: ◎")

print("\n" + "=" * 60)